<a href="https://colab.research.google.com/github/amilafr/algo-python-pro2/blob/main/Web_Quiz_(Complete).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# db_scripts.py

In [ ]:
import sqlite3
from random import randint

db_name = 'quiz.sqlite'
conn = None
cursor = None

def open():
    global conn, cursor
    conn = sqlite3.connect(db_name)
    cursor = conn.cursor()

def close():
    cursor.close()
    conn.close()

def do(query):
    cursor.execute(query)
    conn.commit()

def clear_db():
    ''' deletes all tables '''
    open()
    query = '''DROP TABLE IF EXISTS quiz_content'''
    do(query)
    query = '''DROP TABLE IF EXISTS question'''
    do(query)
    query = '''DROP TABLE IF EXISTS quiz'''
    do(query)
    close()

def create():
    open()
    cursor.execute('''PRAGMA foreign_keys=on''')

    do('''CREATE TABLE IF NOT EXISTS quiz (
            id INTEGER PRIMARY KEY,
            name VARCHAR)'''
    )
    do('''CREATE TABLE IF NOT EXISTS question (
                id INTEGER PRIMARY KEY,
                question VARCHAR,
                answer VARCHAR,
                wrong1 VARCHAR,
                wrong2 VARCHAR,
                wrong3 VARCHAR)'''
    )
    do('''CREATE TABLE IF NOT EXISTS quiz_content (
                id INTEGER PRIMARY KEY,
                quiz_id INTEGER,
                question_id INTEGER,
                FOREIGN KEY (quiz_id) REFERENCES quiz (id),
                FOREIGN KEY (question_id) REFERENCES question (id) )'''
    )
    close()

def show(table):
    query = 'SELECT * FROM ' + table
    open()
    cursor.execute(query)
    print(cursor.fetchall())
    close()

def show_tables():
    show('question')
    show('quiz')
    show('quiz_content')

def add_questions():
    questions = [
        ('How many months in a year have 28 days?', 'All', 'One', 'None','Two'),
        ('What will the green cliff look like if it falls into the Red Sea?', 'Wet', 'Red', 'Will not change', 'Purple'),
        ('Which hand is better to stir tea with?', 'With a spoon', 'Right', 'Left', 'Any'),
        ('What has no length, depth, width, or height, but can be measured?', 'Time', 'Stupidity', 'The sea','Air'),
        ('When is it possible to draw out water with a net?', 'When the water is frozen', 'When there are no fish', 'When the goldfish swim away', 'When the ne breaks'),
        ('What is bigger than an elephant and weighs nothing?', 'Shadow of elephant','A balloon','A parachute', 'A cloud')
    ]
    open()
    cursor.executemany('''INSERT INTO question (question, answer, wrong1, wrong2, wrong3) VALUES (?,?,?,?,?)''', questions)
    conn.commit()
    close()

def add_quiz():
    quizes = [
        ('Quiz 1', ),
        ('Quiz 2', ),
        ('Strange Quiz', )
    ]
    open()
    cursor.executemany('''INSERT INTO quiz (name) VALUES (?)''', quizes)
    conn.commit()
    close()

def add_links():
    open()
    cursor.execute('''PRAGMA foreign_keys=on''')
    query = "INSERT INTO quiz_content (quiz_id, question_id) VALUES (?,?)"
    answer = input("Add a link (y/n)?")
    while answer != 'n':
        quiz_id = int(input("quiz id: "))
        question_id = int(input("question id: "))
        cursor.execute(query, [quiz_id, question_id])
        conn.commit()
        answer = input("Add a link (y/n)?")
    close()


def get_question_after(last_id=0, vict_id=1):
    ''' returns the next question after the question with the passed ID
    for the first question, the default value is passed'''
    open()
    query = '''
    SELECT quiz_content.id, question.question, question.answer, question.wrong1, question.wrong2, question.wrong3
    FROM question, quiz_content
    WHERE quiz_content.question_id == question.id
    AND quiz_content.id > ? AND quiz_content.quiz_id == ?
    ORDER BY quiz_content.id '''
    cursor.execute(query, [last_id, vict_id] )

    result = cursor.fetchone()
    close()
    return result

def get_quises():
    '''returns a list of quizzes (id, name)
    you can only take quizzes that have questions, but for now we choose a simple option '''
    query = 'SELECT * FROM quiz ORDER BY id'
    open()
    cursor.execute(query)
    result = cursor.fetchall()
    close()
    return result

def check_answer(q_id, ans_text):
    query = '''
            SELECT question.answer
            FROM quiz_content, question
            WHERE quiz_content.id = ?
            AND quiz_content.question_id = question.id
        '''
    open()
    cursor.execute(query, str(q_id))
    result = cursor.fetchone()
    close()
    # print(result)
    if result is None:
        return False # cannot find
    else:
        if result[0] == ans_text:
            # print(ans_text)
            return True # the answer matched
        else:
            return False # found but didn't match

def get_quiz_count():
    ''' optional function '''
    query = 'SELECT MAX(quiz_id) FROM quiz_content'
    open()
    cursor.execute(query)
    result = cursor.fetchone()
    close()
    return result

def get_random_quiz_id():
    query = 'SELECT quiz_id FROM quiz_content'
    open()
    cursor.execute(query)
    ids = cursor.fetchall()
    rand_num = randint(0, len(ids) - 1)
    rand_id = ids[rand_num][0]
    close()
    return rand_id

def main():
    clear_db()
    create()
    add_questions()
    add_quiz()
    show_tables()
    add_links()
    show_tables()
    # print(get_question_after(0, 3))
    # print(get_quiz_count())
    # print(get_random_quiz_id())
    pass

if __name__ == "__main__":
    main()

# quiz.py

In [ ]:
import os
from random import shuffle
from flask import Flask, session, request, redirect, render_template, url_for
from db_scripts import get_question_after, get_quises, check_answer

def start_quiz(quiz_id):
    '''creates the desired values ​​in the session dictionary'''
    session['quiz'] = quiz_id
    session['last_question'] = 0
    session['answers'] = 0
    session['total'] = 0

def end_quiz():
    session.clear()

def quiz_form():
    ''' the function receives a list of quizzes from the database and forms a form with a drop-down list '''
    q_list = get_quises()
    return render_template('start.html', q_list=q_list)

def index():
    ''' First page: if you came with a GET request, then select a quiz,
    if POST - then remember the id of the quiz and send questions '''
    if request.method == 'GET':
        # the quiz is not selected, reset the id of the quiz and show the selection form
        start_quiz(-1)
        return quiz_form()
    else:
        # received additional data in the request! We use them:
        quest_id = request.form.get('quiz') #selected quiz number
        start_quiz(quest_id)
        return redirect(url_for('test'))

def save_answers():
    '''receives data from the form, checks if the answer is correct, writes the results to the session'''
    answer = request.form.get('ans_text')
    quest_id = request.form.get('q_id')
    # this question has already been asked:
    session['last_question'] = quest_id
    # increase the question counter:
    session['total'] += 1
    # check if the answer matches the correct one for this id
    if check_answer(quest_id, answer):
        session['answers'] += 1

def question_form(question):
    '''gets the row from the database corresponding to the question, returns the html with the form '''
    # question - result of the get_question_after
    # fields:
            # [0] - quiz question number,
            # [1] - question text,
            # [2] - right answer, [3],[4],[5] - false answers

    # shuffle the answers:
    answers_list = [
        question[2], question[3], question[4], question[5]
    ]
    shuffle(answers_list)
    # pass it to the template, return the result:
    return render_template('test.html', question=question[1], quest_id=question[0], answers_list=answers_list)

def test():
    '''returns the question page'''
    # what if the user without choosing a quiz went directly to address '/test'?
    if not ('quiz' in session) or int(session['quiz']) < 0:
        return redirect(url_for('index'))
    else:
        # if we received data, then we need to read it and update the information:
        if request.method == 'POST':
            save_answers()
        # in any case, deal with the current question id
        next_question = get_question_after(session['last_question'], session['quiz'])
        if next_question is None or len(next_question) == 0:
            # the questions are over:
            return redirect(url_for('result'))
        else:
            return question_form(next_question)

def result():
    html = render_template('result.html', right=session['answers'], total=session['total'])
    end_quiz()
    return html

folder = os.getcwd() # remember the current working folder
# Create a web application object:
app = Flask(__name__, template_folder=folder, static_folder=folder)
app.add_url_rule('/', 'index', index, methods=['post', 'get'])   # creates rule for URL '/'
app.add_url_rule('/test', 'test', test, methods=['post', 'get']) # creates rule for URL '/test'
app.add_url_rule('/result', 'result', result) # creates rule for URL '/test'
#Setting up the encryption key:
app.config['SECRET_KEY'] = 'ThisIsSecretSecretSecretLife'

if __name__ == "__main__":
    #Launching web server:
    app.run()

# start.html

In [ ]:
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
    <title>Quiz</title>
</head>
<body>
<h2>Choose quiz:</h2>
    <form method="post" action="/">
        <select name="quiz">
            {% for id, name in q_list %}
                <option value="{{id}}">{{name}}</option>
            {% endfor %}
        </select>
        <p><input type="submit" value="Choose"> </p>
    </form>
</body>
</html>

# test.html

In [ ]:
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Quiz</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
    {% if question %}
        <h1>{{question}}</h1>
    {% endif %}
    <form method="post" action="test">
        <input type="hidden" name="q_id" value="{{quest_id}}">
        <div class="outer-div">
        {% for answer in answers_list %}
            <div class="form_radio_btn">
                <input type="radio" name="ans_text" value="{{ answer }}">{{ answer }}
            </div>
        {% endfor %}
        <div class="inner-div">
            <p>
            <input class="btn" type="submit" value="Answer">
            </p>
        </div>
        </div>
    </form>
</body>
</html>

# result.html

In [ ]:
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Title</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<h1>Your result:</h1>
<div class="outer-div"><h3>{{ right }} right answers. Total answers {{ total }}</h3></div>
<a href="{{url_for('index')}}" class="button"> Again </a>
</body>
</html>